# 05. vLLM으로 치환

- 직접 만든 scheduler와 worker를 vLLM으로 치환
- Ubuntu RTX 환경에서 실행

## 환경 제한

vLLM library mode는 Ubuntu RTX 환경에서 `uv sync --dev --extra gpu` 후 실행한다. macOS에서는 이 코드 셀을 읽고 server mode client만 실행한다.

## 구현 코드

셀을 실행해 vLLM 기반 engine과 endpoint를 직접 정의한다.

In [ ]:
"""v4: delegate batching and scheduling to vLLM (library mode).

Ubuntu + NVIDIA GPU only. On Apple Silicon use compare_openai.py against a
remote or containerised vLLM server instead.
"""

import os

from fastapi import FastAPI
from pydantic import BaseModel
from vllm import LLM, SamplingParams

MODEL_NAME = os.getenv("MODEL_NAME", "facebook/opt-125m")
MAX_TOKENS = int(os.getenv("MAX_TOKENS", "20"))
MAX_NUM_SEQS = int(os.getenv("MAX_NUM_SEQS", "16"))
GPU_MEMORY_UTILIZATION = float(os.getenv("GPU_MEMORY_UTILIZATION", "0.6"))


class LLMEngine:
  def __init__(self):
    self.vllm_model = LLM(
      model=MODEL_NAME,
      max_num_seqs=MAX_NUM_SEQS,
      gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    )
    self.max_tokens = MAX_TOKENS

  def generate_vllm(self, prompts: list[str]) -> list[str]:
    sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=self.max_tokens)
    outputs = self.vllm_model.generate(prompts, sampling_params)
    return [output.outputs[0].text for output in outputs]


app = FastAPI(title="ch03 v4 - vLLM")
_llm = None


def get_llm() -> LLMEngine:
  global _llm
  if _llm is None:
    _llm = LLMEngine()
  return _llm


class BatchGenerateRequest(BaseModel):
  prompts: list[str]


class BatchGenerateResponse(BaseModel):
  generated_texts: list[str]


@app.post("/generate_vllm", response_model=BatchGenerateResponse)
async def generate_vllm(request: BatchGenerateRequest):
  return BatchGenerateResponse(generated_texts=get_llm().generate_vllm(request.prompts))


@app.get("/healthz")
async def healthz():
  return {"status": "ok", "model": MODEL_NAME, "max_num_seqs": MAX_NUM_SEQS}

## vLLM engine 직접 실행

여러 prompt가 engine 내부 scheduler로 전달된다.

In [ ]:
engine = LLMEngine()
engine.generate(["Hello, I am", "The weather is"])

## 퀴즈

코드를 다시 보지 않고 먼저 답해본다.

1. `max_num_seqs`는 직접 구현의 어떤 제한값과 대응하는가?
2. library mode와 standalone server mode의 운영상 차이는 무엇인가?

<details>
<summary>정답과 해설 보기</summary>

1. 동시에 active batch에 포함할 sequence 수를 제한하는 `batch_size`와 대응한다. 실제 vLLM에서는 scheduler가 이 상한 안에서 sequence를 동적으로 구성한다.
2. library mode는 애플리케이션 process 안에서 engine을 직접 호출해 제어하기 쉽다. server mode는 별도 process와 OpenAI 호환 API로 분리되어 독립 배포·스케일링과 다언어 client 연결에 유리하다.

</details>